# Train `tre-vi-0.6b` trên Kaggle (miễn phí)

**Yêu cầu:** tài khoản Kaggle đã verify phone, bật GPU: Settings → Accelerator → **GPU T4 x2** (hoặc P100).

**Kết quả:** adapter LoRA `tre-adapter-vi` trong `/kaggle/working/` → tải về, merge + convert GGUF ở local bằng `tre train export`.

Provenance trung thực: base = `Qwen/Qwen3-0.6B` (Apache-2.0), adapter = Tre. Dataset: Apache-2.0 + CC-BY-4.0.

In [ ]:
# 1) Cài deps + clone repo (pin commit để reproduce)
!pip install -q "transformers>=4.57" "trl>=0.21" "peft>=0.17" "datasets>=3.6" accelerate sentencepiece pyyaml
!git clone --depth 1 https://github.com/tang-vu/tre-llm.git /kaggle/working/tre-llm
%cd /kaggle/working/tre-llm
!pip install -q -e . --no-deps

In [ ]:
# 2) Kiểm tra GPU
import torch

print(torch.cuda.is_available(), torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")
assert torch.cuda.is_available(), "Chưa bật GPU — Settings → Accelerator"

In [ ]:
# 3) Chuẩn bị + validate dữ liệu (HF datasets, license sạch, leakage check vs eval suite)
!tre train prepare --recipe recipes/tre-vi-0.6b/recipe.yaml
!tre train validate --data recipes/tre-vi-0.6b/prepared.jsonl

In [ ]:
# 4) Preflight — trên Kaggle VRAM đủ nên đi qua nhánh GPU thật
!tre train preflight --recipe recipes/tre-vi-0.6b/recipe.yaml

In [ ]:
# 5) Train — ~3.5k rows, LoRA r=16, 2 epochs, seq 1024.
#    Ước lượng trên T4: ~45-90 phút (trong quota 30h/tuần của Kaggle).
!tre train run --recipe recipes/tre-vi-0.6b/recipe.yaml

In [ ]:
# 6) Đóng gói adapter để tải về (Output tab → tre-adapter-vi.zip)
import pathlib
import shutil

out = pathlib.Path('/kaggle/working/tre-adapter-vi')
shutil.copytree('training-out/tre-vi-0.6b/adapter', out, dirs_exist_ok=True)
shutil.copy('training-out/tre-vi-0.6b/run-report.json', out / 'run-report.json')
shutil.make_archive('/kaggle/working/tre-adapter-vi', 'zip', out)
print('adapter tại /kaggle/working/tre-adapter-vi.zip')

## Sau khi tải adapter về máy

```bash
# merge + convert + quantize (cần llama.cpp source đúng build runtime)
tre train export --adapter <đường-dẫn>/tre-adapter-vi --out ~/.local/share/tre-llm/models/tre-vi-0.6b-q4_k_m.gguf
tre models import ~/.local/share/tre-llm/models/tre-vi-0.6b-q4_k_m.gguf --id tre-vi-0.6b-q4_k_m

# đo trước/sau — trung thực
tre eval --split dev --model qwen3-0.6b-base-q4_k_m   # baseline
tre eval --split dev --model tre-vi-0.6b-q4_k_m       # sau adapter
```